# Tema 6, Exercițiul 2
## Cât de repede își primesc clienții mâncarea? (simulare Monte Carlo)

Un fast-food cu o singură casă și o singură stație de gătit. Clienții sosesc după un proces Poisson cu intensitatea $\lambda = 20$ clienți/oră. Fiecare client trece întâi pe la casă (timp de plată exponențial cu media $2$ minute), apoi comanda intră în coada stației de gătit (timp de preparare exponențial cu media $\alpha$ minute).

### Ipoteze și notații

- proces Poisson de sosiri $\Rightarrow$ momentele de sosire în $[0, 60]$ min sunt uniforme, iar numărul lor este $N \sim \text{Poisson}(20)$;
- două stații **FIFO** în serie (tandem): casa, apoi gătitul, fiecare cu un singur server;
- timpii de plată $\sim \text{Exp}(\text{media} = 2)$, timpii de preparare $\sim \text{Exp}(\text{media} = \alpha)$, independenți între ei și de sosiri;
- la începutul orei sistemul este gol; urmărim până la servirea completă clienții sosiți în $[0, 60]$, chiar dacă termină după minutul $60$.

Pentru clientul $i$ (în ordinea sosirii) notăm: $a_i$ momentul sosirii, $c_i$ momentul la care termină la casă, $f_i$ momentul la care primește mâncarea.

## a) Simularea unei ore

Cele două cozi sunt FIFO cu câte un server. Le tratăm urmărind momentul la care fiecare server devine liber:

- **casa**: clientul $i$ începe plata la $\max(a_i,\ \text{casă liberă})$ și termină la $c_i$;
- **gătit**: comanda lui $i$ intră la gătit abia după ce a terminat la casă; începe la $\max(c_i,\ \text{gătit liber})$ și se termină la $f_i$.

Pentru că toți timpii de servire sunt pozitivi și clienții sunt procesați în ordinea sosirii, ordinea la gătit coincide cu ordinea la casă, deci un singur `for` peste clienți, în ordinea sosirii, este suficient.

In [1]:
import numpy as np

LAMBDA = 20        # clienti/ora
ORA = 60.0         # minute
MEDIA_PLATA = 2.0  # minute (medie exponentiala la casa)


def simuleaza_ora(alpha, rng):
    '''Simuleaza o ora. Returneaza timpii sosire -> primire mancare (minute).'''
    n = rng.poisson(LAMBDA)                  # numarul de clienti din aceasta ora
    if n == 0:
        return np.array([])
    sosiri = np.sort(rng.uniform(0, ORA, n))       # momente de sosire, FIFO
    plata = rng.exponential(MEDIA_PLATA, n)        # timp la casa
    preparare = rng.exponential(alpha, n)          # timp la gatit

    primire = np.empty(n)
    casa_libera = 0.0
    gatit_liber = 0.0
    for i in range(n):
        termina_casa = max(sosiri[i], casa_libera) + plata[i]
        casa_libera = termina_casa
        termina_gatit = max(termina_casa, gatit_liber) + preparare[i]
        gatit_liber = termina_gatit
        primire[i] = termina_gatit

    return primire - sosiri  # timp total sosire -> mancare, per client


# o rulare exemplu (alpha = 2 min)
rng = np.random.default_rng(0)
exemplu = simuleaza_ora(2.0, rng)
print(f"clienti in ora exemplu: {len(exemplu)}")
print(f"timp maxim sosire->mancare: {exemplu.max():.2f} min")
print(f"timp mediu sosire->mancare: {exemplu.mean():.2f} min")

clienti in ora exemplu: 22
timp maxim sosire->mancare: 17.29 min
timp mediu sosire->mancare: 7.21 min


## b) Probabilitatea ca toți clienții să fie serviți în ≤ 15 min și căutarea lui $\alpha$

Condiția se referă la *toți* clienții dintr-o oră, nu la un client ales aleatoriu. Deci un experiment (o oră) este un succes doar dacă timpul maxim sosire→mâncare din acea oră este $\le 15$ min:

$$p(\alpha) = P\big(\max_i (f_i - a_i) \le 15\big).$$

Timpul de preparare al fiecărui client crește cu $\alpha$, iar într-o coadă FIFO timpii de servire mai mari nu pot scădea momentul de primire al niciunui client. Deci $\max_i(f_i - a_i)$ crește cu $\alpha$, iar $p(\alpha)$ este descrescătoare. Prin urmare, pentru *orice* $\alpha \ge 0$:

$$p(\alpha) \le \lim_{\alpha \to 0^+} p(\alpha) = p(0),$$

unde $p(0)$ corespunde gătitului instantaneu (preparare $= 0$), adică sistemul se reduce la o singură coadă la casă.

Estimăm întâi această margine superioară $p(0)$, apoi $p(\alpha)$ pe o grilă, și căutăm cel mai mare $\alpha$ cu $p(\alpha) \ge 0.95$.

In [2]:
# margine superioara: p(0) = gatit instantaneu (doar coada de la casa)
def prob_toti_sub_15_alpha0(n_sim, seed):
    rng = np.random.default_rng(seed)
    succese = 0
    for _ in range(n_sim):
        n = rng.poisson(LAMBDA)
        if n == 0:
            succese += 1
            continue
        sosiri = np.sort(rng.uniform(0, ORA, n))
        plata = rng.exponential(MEDIA_PLATA, n)
        casa_libera = 0.0
        worst = 0.0
        for i in range(n):
            termina_casa = max(sosiri[i], casa_libera) + plata[i]
            casa_libera = termina_casa
            worst = max(worst, termina_casa - sosiri[i])  # fara timp de gatit
        succese += (worst <= 15)
    return succese / n_sim


p0 = prob_toti_sub_15_alpha0(50_000, 42)
print(f"p(0) (gatit instantaneu, margine superioara) = {p0:.3f}")
print(f"=> p(alpha) <= {p0:.3f} < 0.95 pentru orice alpha >= 0")

p(0) (gatit instantaneu, margine superioara) = 0.810
=> p(alpha) <= 0.810 < 0.95 pentru orice alpha >= 0


In [3]:
def prob_toti_sub_15(alpha, n_sim, seed):
    '''Fractia de ore in care TOTI clientii primesc mancarea in <= 15 min.'''
    rng = np.random.default_rng(seed)
    succese = 0
    for _ in range(n_sim):
        timpi = simuleaza_ora(alpha, rng)
        # ora goala: conditia e vacuos adevarata; altfel conteaza cel mai lent client
        if timpi.size == 0 or timpi.max() <= 15:
            succese += 1
    return succese / n_sim


N_SIM = 20_000
SEED = 42

grila_alpha = np.arange(0.25, 4.01, 0.25)
prob = np.array([prob_toti_sub_15(a, N_SIM, SEED) for a in grila_alpha])

for a, p in zip(grila_alpha, prob):
    print(f"alpha = {a:4.2f} min  ->  P(toti <= 15 min) = {p:.3f}")

alpha = 0.25 min  ->  P(toti <= 15 min) = 0.796
alpha = 0.50 min  ->  P(toti <= 15 min) = 0.782
alpha = 0.75 min  ->  P(toti <= 15 min) = 0.762
alpha = 1.00 min  ->  P(toti <= 15 min) = 0.733
alpha = 1.25 min  ->  P(toti <= 15 min) = 0.689
alpha = 1.50 min  ->  P(toti <= 15 min) = 0.624
alpha = 1.75 min  ->  P(toti <= 15 min) = 0.540
alpha = 2.00 min  ->  P(toti <= 15 min) = 0.445
alpha = 2.25 min  ->  P(toti <= 15 min) = 0.354
alpha = 2.50 min  ->  P(toti <= 15 min) = 0.271
alpha = 2.75 min  ->  P(toti <= 15 min) = 0.205
alpha = 3.00 min  ->  P(toti <= 15 min) = 0.152
alpha = 3.25 min  ->  P(toti <= 15 min) = 0.112
alpha = 3.50 min  ->  P(toti <= 15 min) = 0.083
alpha = 3.75 min  ->  P(toti <= 15 min) = 0.060
alpha = 4.00 min  ->  P(toti <= 15 min) = 0.045


Probabilitatea scade monoton cu $\alpha$ (gătit mai lent $\Rightarrow$ cozi mai lungi). Verificăm dacă pragul de $95\%$ este atins pentru vreun $\alpha$ de pe grilă.

In [4]:
valide = grila_alpha[prob >= 0.95]
if valide.size > 0:
    print(f"cel mai mare alpha cu P >= 0.95: {valide.max():.2f} min")
else:
    print("niciun alpha de pe grila nu atinge P >= 0.95")
    print(f"maximul probabilitatii este {prob.max():.3f}, la alpha = {grila_alpha[prob.argmax()]:.2f}")

niciun alpha de pe grila nu atinge P >= 0.95
maximul probabilitatii este 0.796, la alpha = 0.25


**Rezultat.** Marginea superioară $p(0) \approx 0.81$ este deja sub $0.95$, deci prin monotonie nu există niciun $\alpha \ge 0$ pentru care $p(\alpha) \ge 0.95$.

Chiar cu gătit instantaneu, clientul cel mai întârziat dintr-o oră aglomerată depășește frecvent $15$ min doar din așteptarea la casă.

Pentru punctele următoare raportăm pentru valoarea cea mai apropiată de obiectiv de pe grilă, $\alpha^\star = 0.25$ min.

## c) Timpul mediu sosire → primirea mâncării pentru $\alpha^\star$

Estimăm, pentru $\alpha^\star = 0.25$ min, timpul mediu dintre sosire și primirea mâncării (mediat peste toți clienții din toate orele simulate). Raportăm separat și timpul mediu petrecut efectiv în așteptare (fără cei doi timpi de servire).

In [5]:
def medii_timpi(alpha, n_sim, seed):
    rng = np.random.default_rng(seed)
    total, asteptare = [], []
    for _ in range(n_sim):
        n = rng.poisson(LAMBDA)
        if n == 0:
            continue
        sosiri = np.sort(rng.uniform(0, ORA, n))
        plata = rng.exponential(MEDIA_PLATA, n)
        preparare = rng.exponential(alpha, n)
        casa_libera = gatit_liber = 0.0
        for i in range(n):
            termina_casa = max(sosiri[i], casa_libera) + plata[i]
            casa_libera = termina_casa
            termina_gatit = max(termina_casa, gatit_liber) + preparare[i]
            gatit_liber = termina_gatit
            t_total = termina_gatit - sosiri[i]
            total.append(t_total)
            asteptare.append(t_total - plata[i] - preparare[i])  # fara servire
    return np.mean(total), np.mean(asteptare)


alpha_star = 0.25
m_total, m_asteptare = medii_timpi(alpha_star, N_SIM, SEED)
print(f"alpha* = {alpha_star} min")
print(f"timp mediu sosire -> mancare: {m_total:.2f} min")
print(f"timp mediu doar in asteptare (fara servire): {m_asteptare:.2f} min")

alpha* = 0.25 min
timp mediu sosire -> mancare: 5.26 min
timp mediu doar in asteptare (fara servire): 3.01 min


La $\alpha^\star = 0.25$ min, timpul mediu sosire→mâncare este de câteva minute, iar cea mai mare parte provine din așteptarea la coada de la casă (nu din servirea propriu-zisă). Media rămâne mult sub $15$ min — problema de la b) nu este media, ci clientul *cel mai întârziat* dintr-o oră aglomerată.

## d) Reproducibilitate și metoda de căutare

- **Număr de simulări:** `N_SIM = 20_000` ore per valoare de $\alpha$.
- **Sămânța aleatoare:** `SEED = 42`, printr-un `numpy.random.default_rng(seed)` reinițializat pentru fiecare $\alpha$, astfel încât comparația între valorile $\alpha$ să folosească aceleași fluxuri aleatoare.
- **Metoda de căutare a lui $\alpha$:** grilă echidistantă `np.arange(0.25, 4.01, 0.25)`; pentru fiecare $\alpha$ estimăm $p(\alpha)$ și reținem cel mai mare $\alpha$ cu $p(\alpha) \ge 0.95$. Cum $p$ este descrescătoare în $\alpha$, grila este suficientă pentru a localiza pragul (aici: inexistent).

Estimările se pot modifica de la o rulare la alta pentru că sunt aproximări **Monte Carlo**: fiecare $p(\alpha)$ este o frecvență pe un eșantion finit de ore, cu eroare de ordinul $\sqrt{p(1-p)/N_\text{sim}}$ (aici $\lesssim 0.004$). Schimbarea seminței sau a numărului de simulări deplasează ușor valorile, fără a schimba concluzia calitativă.